# pinkln Agent Architecture System - Comprehensive Demo

**"Insanely Great AI Systems Through Elegant Orchestration"**

This notebook demonstrates the full capabilities of the pinkln Agent Architecture System running on Google Cloud Vertex AI Workbench.

## What You'll Learn

1. Using individual Skills for specific tasks
2. Leveraging Agents for complex workflows
3. Multi-Agent collaboration and debate
4. Adaptive reasoning strategies
5. Cost optimization techniques
6. Production deployment patterns

## Prerequisites

Make sure you've completed `vertex_workbench_setup.ipynb` first!

## Setup and Imports

In [ ]:
import os
import sys
from pathlib import Path

import yaml
from anthropic import AnthropicVertex
from google.cloud import aiplatform

# Load configuration
config_path = Path("vertex_config.yaml")
if config_path.exists():
    with open(config_path) as f:
        config = yaml.safe_load(f)
else:
    config = {
        "google_cloud": {
            "project_id": os.getenv("GOOGLE_CLOUD_PROJECT"),
            "location": "us-central1",
        },
        "claude": {"default_model": "claude-3-5-sonnet-v2@20241022"},
    }

PROJECT_ID = config["google_cloud"]["project_id"]
LOCATION = config["google_cloud"]["location"]
MODEL = config["claude"]["default_model"]

# Initialize Vertex AI
aiplatform.init(project=PROJECT_ID, location=LOCATION)

# Add pinkln to path
sys.path.insert(0, str(Path.cwd()))

from pinkln.core.master_system import PnklnOS

print("✓ Configuration loaded")
print(f"  Project: {PROJECT_ID}")
print(f"  Location: {LOCATION}")
print(f"  Model: {MODEL}")

## Part 1: Understanding Complexity-Based Reasoning

The pinkln system automatically selects the best reasoning strategy based on problem complexity.

In [ ]:
# Initialize pinkln OS
pinkln_os = PnklnOS()

# Test different complexity levels
problems = [
    {
        "name": "Simple Calculation",
        "problem": "What is 25% of $1000?",
        "expected_strategy": "chain_of_thought",
    },
    {
        "name": "Medium Complexity",
        "problem": "Should I build or buy a CRM system? I have a team of 5 developers and $50K budget.",
        "expected_strategy": "tree_of_thoughts",
    },
    {
        "name": "High Complexity",
        "problem": """Design a comprehensive AI monetization strategy for a B2B SaaS company.
        Consider: pricing models, feature tiers, upsell opportunities, market positioning,
        competitive analysis, and 18-month revenue projections. What should we prioritize?""",
        "expected_strategy": "multi_agent_debate",
    },
]

print("Complexity Assessment Results:\n" + "=" * 70)
for p in problems:
    complexity = pinkln_os.assess_complexity(p["problem"])
    strategy = pinkln_os.select_reasoning_strategy(complexity)

    print(f"\n{p['name']}:")
    print(f"  Complexity Score: {complexity:.2f}")
    print(f"  Selected Strategy: {strategy}")
    print(f"  Expected Strategy: {p['expected_strategy']}")
    print(f"  Match: {'✓' if strategy == p['expected_strategy'] else '✗'}")

print("\n" + "=" * 70)

## Part 2: Using the Vertex-Integrated Agent

Let's create a helper class that integrates pinkln with Claude on Vertex AI.

In [ ]:
class VertexPnklnAgent:
    """Integrated pinkln Agent with Vertex AI Claude."""

    def __init__(
        self, project_id: str, location: str, model: str = "claude-3-5-sonnet-v2@20241022"
    ):
        self.client = AnthropicVertex(region=location, project_id=project_id)
        self.model = model
        self.pinkln_os = PnklnOS()
        self.usage_stats = {"total_input_tokens": 0, "total_output_tokens": 0, "total_calls": 0}

    def execute(self, challenge: str, role: str = "Agent", max_tokens: int = 4096) -> dict:
        """Execute a challenge with pinkln orchestration."""
        # Assess complexity
        complexity = self.pinkln_os.assess_complexity(challenge)
        strategy = self.pinkln_os.select_reasoning_strategy(complexity)

        # Create role-specific prompt
        system_prompt = self.pinkln_os.create_agent_prompt(
            agent_role=role,
            task=challenge,
            reasoning_strategy=strategy,
            complexity_score=f"{complexity:.2f}",
        )

        # Call Claude via Vertex AI
        response = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            system=system_prompt,
            messages=[{"role": "user", "content": challenge}],
        )

        # Track usage
        self.usage_stats["total_input_tokens"] += response.usage.input_tokens
        self.usage_stats["total_output_tokens"] += response.usage.output_tokens
        self.usage_stats["total_calls"] += 1

        return {
            "solution": response.content[0].text,
            "complexity": complexity,
            "strategy": strategy,
            "role": role,
            "usage": {
                "input_tokens": response.usage.input_tokens,
                "output_tokens": response.usage.output_tokens,
            },
        }

    def get_usage_summary(self) -> dict:
        """Get token usage summary and cost estimate."""
        # Claude 3.5 Sonnet pricing on Vertex AI (as of 2024)
        input_cost_per_mtok = 3.00  # $3 per million input tokens
        output_cost_per_mtok = 15.00  # $15 per million output tokens

        input_cost = (self.usage_stats["total_input_tokens"] / 1_000_000) * input_cost_per_mtok
        output_cost = (self.usage_stats["total_output_tokens"] / 1_000_000) * output_cost_per_mtok

        return {
            **self.usage_stats,
            "estimated_cost_usd": input_cost + output_cost,
            "breakdown": {"input_cost_usd": input_cost, "output_cost_usd": output_cost},
        }


# Initialize agent
agent = VertexPnklnAgent(project_id=PROJECT_ID, location=LOCATION, model=MODEL)
print("✓ VertexPnklnAgent initialized")

## Part 3: Real-World Example - Revenue Optimization

Let's use the agent to analyze a real revenue optimization scenario.

In [ ]:
revenue_challenge = """
I run a SaaS business with the following metrics:
- 1,200 monthly active users
- $29/month subscription price
- 85% monthly retention rate
- $150 customer acquisition cost
- 5% free-to-paid conversion rate

Our product offers project management features. Competitors charge $15-$50/month.
I suspect we're leaving money on the table. What are the top 3 revenue opportunities,
and what specific actions should I take this week?
"""

print("Executing Revenue Optimization Challenge...\n")
result = agent.execute(revenue_challenge, role="Monetization Architect")

print("=" * 70)
print("REVENUE OPTIMIZATION ANALYSIS")
print("=" * 70)
print(f"\nComplexity: {result['complexity']:.2f}")
print(f"Strategy: {result['strategy']}")
print(f"Role: {result['role']}")
print(f"\n{result['solution']}")
print("\n" + "=" * 70)
print(f"Tokens: {result['usage']['input_tokens']} in / {result['usage']['output_tokens']} out")
print("=" * 70)

## Part 4: Design Critique Example

Using the Design Critic role to analyze and improve a user interface.

In [ ]:
design_challenge = """
Review our checkout flow:

1. User clicks "Buy Now" button
2. Redirected to form with 12 fields: name, email, phone, company, address, city,
   state, zip, country, card number, CVV, expiry
3. Submit button at bottom
4. After submit, shows "Processing..." for 3-5 seconds
5. Redirects to success page

Our conversion rate is 12%. What's wrong and how do we fix it?
"""

print("Executing Design Critique...\n")
result = agent.execute(design_challenge, role="Design Critic")

print("=" * 70)
print("DESIGN CRITIQUE ANALYSIS")
print("=" * 70)
print(f"\n{result['solution']}")
print("\n" + "=" * 70)

## Part 5: Multi-Agent Collaboration Pattern

Simulating a council of experts for complex decisions.

In [ ]:
# Simulate multi-agent debate by running the same challenge through different personas
complex_challenge = """
Should our startup pivot from B2C to B2B?

Context:
- Current: 50K free users, 500 paying ($10/mo)
- Runway: 8 months
- Team: 4 engineers, 1 designer, 1 PM
- We've received interest from 3 enterprise companies willing to pay $2K/month
- But our product needs significant changes for B2B
"""

personas = [
    {"role": "Optimistic Growth Strategist", "focus": "opportunities and growth potential"},
    {"role": "Risk-Averse CFO", "focus": "financial risks and runway preservation"},
    {"role": "Pragmatic Product Leader", "focus": "execution feasibility and team capacity"},
]

print("Running Multi-Agent Debate...\n")
print("=" * 70)
print("COUNCIL OF EXCELLENCE - PIVOT DECISION")
print("=" * 70)

perspectives = []
for persona in personas:
    enhanced_challenge = f"""
{complex_challenge}

Provide your perspective focusing on {persona["focus"]}.
Be concise but thorough. End with a clear recommendation (Yes/No/Conditional).
"""
    result = agent.execute(enhanced_challenge, role=persona["role"])
    perspectives.append(result)

    print(f"\n{'=' * 70}")
    print(f"{persona['role'].upper()}")
    print(f"{'=' * 70}")
    print(result["solution"])

print(f"\n{'=' * 70}")
print("DEBATE COMPLETE")
print(f"{'=' * 70}")

## Part 6: Synthesis - Combine Perspectives

Now let's synthesize all perspectives into a final recommendation.

In [ ]:
# Combine all perspectives for synthesis
synthesis_challenge = f"""
Three experts have analyzed whether our startup should pivot from B2C to B2B.
Here are their perspectives:

OPTIMISTIC GROWTH STRATEGIST:
{perspectives[0]["solution"]}

RISK-AVERSE CFO:
{perspectives[1]["solution"]}

PRAGMATIC PRODUCT LEADER:
{perspectives[2]["solution"]}

Synthesize these perspectives into:
1. A clear final recommendation (Yes/No/Hybrid approach)
2. Top 3 action items for next week
3. Key risks to monitor
4. Success metrics to track
"""

print("Synthesizing perspectives...\n")
synthesis = agent.execute(
    synthesis_challenge, role="Executive Decision Synthesizer", max_tokens=2048
)

print("=" * 70)
print("FINAL SYNTHESIS")
print("=" * 70)
print(f"\n{synthesis['solution']}")
print("\n" + "=" * 70)

## Part 7: Usage Statistics and Cost Analysis

In [ ]:
usage = agent.get_usage_summary()

print("=" * 70)
print("SESSION USAGE SUMMARY")
print("=" * 70)
print(f"\nTotal API Calls: {usage['total_calls']}")
print(f"Total Input Tokens: {usage['total_input_tokens']:,}")
print(f"Total Output Tokens: {usage['total_output_tokens']:,}")
print(f"Total Tokens: {usage['total_input_tokens'] + usage['total_output_tokens']:,}")
print("\nCost Breakdown:")
print(f"  Input Cost: ${usage['breakdown']['input_cost_usd']:.4f}")
print(f"  Output Cost: ${usage['breakdown']['output_cost_usd']:.4f}")
print(f"  Total Estimated Cost: ${usage['estimated_cost_usd']:.4f}")
print(
    f"\nAverage tokens per call: {(usage['total_input_tokens'] + usage['total_output_tokens']) / usage['total_calls']:.0f}"
)
print("=" * 70)

## Part 8: Best Practices for Production

### Cost Optimization Tips

1. **Use appropriate models**: Use Claude 3.5 Haiku for simple tasks, Sonnet for complex reasoning
2. **Implement semantic caching**: Cache similar queries to reduce token usage by 40-60%
3. **Optimize max_tokens**: Set appropriate limits based on expected response length
4. **Batch processing**: Process multiple items in a single call when possible
5. **Monitor usage**: Track token consumption and set budgets

### Production Deployment Checklist

- [ ] Enable Workload Identity for secure authentication
- [ ] Set up proper IAM roles and permissions
- [ ] Implement error handling and retries
- [ ] Add logging and monitoring
- [ ] Configure auto-shutdown for idle instances
- [ ] Use version control for prompts and configurations
- [ ] Implement rate limiting
- [ ] Set up cost alerts in Google Cloud Console

## Part 9: Advanced Pattern - Iterative Refinement

Demonstrate the "Iterate to Excellence" pattern from pinkln philosophy.

In [ ]:
def iterate_to_excellence(agent, challenge, role, iterations=3):
    """
    Iteratively refine a solution using the pinkln "Iterate Relentlessly" principle.
    """
    print(f"Iterating to Excellence ({iterations} rounds)...\n")

    # Initial solution
    result = agent.execute(challenge, role=role)
    print(f"{'=' * 70}")
    print("ITERATION 1 - Initial Solution")
    print(f"{'=' * 70}")
    print(result["solution"][:500] + "...\n")

    # Refinement iterations
    for i in range(2, iterations + 1):
        critique_challenge = f"""
Here's a solution to a challenge:

{result["solution"]}

Apply the pinkln principles to critique and improve this:
1. What's missing or could be clearer?
2. How can we simplify ruthlessly?
3. What details need more obsession?
4. Provide an improved version.
"""
        result = agent.execute(critique_challenge, role=f"{role} (Refinement Round {i})")
        print(f"{'=' * 70}")
        print(f"ITERATION {i} - Refinement")
        print(f"{'=' * 70}")
        print(result["solution"][:500] + "...\n")

    return result


# Example: Iterate on a prompt design
prompt_challenge = """
Create a prompt for an AI assistant that helps users write better cold emails
for B2B sales. The prompt should ensure the AI produces personalized,
compelling emails with high response rates.
"""

final_result = iterate_to_excellence(agent, prompt_challenge, "Prompt Craftsman", iterations=2)

print("=" * 70)
print("FINAL EXCELLENCE VERSION")
print("=" * 70)
print(final_result["solution"])

## Summary

You've now learned:

1. ✓ How complexity-based reasoning works in pinkln
2. ✓ Integration with Claude on Vertex AI
3. ✓ Real-world revenue optimization analysis
4. ✓ Design critique workflows
5. ✓ Multi-agent collaboration patterns
6. ✓ Perspective synthesis for complex decisions
7. ✓ Cost tracking and optimization
8. ✓ Iterative refinement to excellence

## Next Steps

1. Customize agents for your specific use cases
2. Build custom skills in `pinkln/skills/`
3. Implement production error handling
4. Set up monitoring and alerting
5. Explore the Superpowers Marketplace for additional components

**Remember:** "The people who are crazy enough to think they can change the world are the ones who do." 🚀